# LAS-Mamba-GNN Training Notebook



## Setup

In [1]:
import sys
import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import random
from tqdm import tqdm

# Find project root - works in Jupyter and script
cwd = Path.cwd()

possible_roots = [
    cwd,
    cwd / "notebooks",
    cwd / "..",
    cwd / ".." / "notebooks",
]

for root in possible_roots:
    if (root / "data").exists() and (root / "src").exists():
        ROOT_DIR = root
        break
else:
    for parent in [cwd] + list(cwd.parents):
        if (parent / "data").exists() and (parent / "src").exists():
            ROOT_DIR = parent
            break
    else:
        ROOT_DIR = cwd

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

os.chdir(ROOT_DIR)
print(f"Working directory: {os.getcwd()}")

from src.dataset.elliptic_dataset import FastEllipticDataset
from src.models.las_mamba_gnn import create_las_mamba_gnn
from src.models.loss import get_loss_function
from src.training.trainer import OptimizedTrainer, get_local_edge_index
from src.utils.config import MODEL_CONFIG, TRAINING_CONFIG, LOSS_CONFIG
from src.utils.metrics import compute_metrics

print(f"   Root directory: {ROOT_DIR}")
print(f"   PyTorch version: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")

Working directory: /home/thanhhai/Downloads/aml-elliptic2-detection-main
   Root directory: /home/thanhhai/Downloads/aml-elliptic2-detection-main/notebooks/..
   PyTorch version: 2.5.1+cpu
   CUDA available: False


## Configuration

In [2]:
# Training configuration
CONFIG = {
    # Data paths
    'graph_dir': 'data/processed/graph',
    'sequences_dir': 'data/processed/sequences',
    'index_dir': 'data/processed/index',
    
    # Model
    'use_gnn': False,
    'use_las': False,
    'use_mamba': True,
    
    # Training
    'device': 'cpu',  # Changed from 'cpu' to 'cuda' for GPU
    'batch_size': 256,  # Increased from 128 for GPU efficiency
    'num_epochs': 100,
    'learning_rate': 5e-4,  # Increased from 1e-5 to 5e-4
    'weight_decay': 1e-4,
    'num_workers': 4,  # Increased from 0 for parallel data loading
    
    # Loss
    'loss_type': 'focal',  # 'weighted_ce' or 'focal'
    'class_weights': [1.0, 5000.0],
    'threshold': 0.15,
    'focal_gamma': 1.0,
    'focal_alpha': 0.75,
    
    # Early stopping
    'early_stopping_patience': 100,
    'val_interval': 1,
    
    # Reproducibility
    'seed': 42,
}

def set_seed(seed):
    """Set random seed for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])


## Load Dataset

In [3]:
print("Loading datasets")

train_dataset = FastEllipticDataset(
    graph_dir=CONFIG['graph_dir'],
    sequences_dir=CONFIG['sequences_dir'],
    index_dir=CONFIG['index_dir'],
    split='train'
)

val_dataset = FastEllipticDataset(
    graph_dir=CONFIG['graph_dir'],
    sequences_dir=CONFIG['sequences_dir'],
    index_dir=CONFIG['index_dir'],
    split='val'
)

test_dataset = FastEllipticDataset(
    graph_dir=CONFIG['graph_dir'],
    sequences_dir=CONFIG['sequences_dir'],
    index_dir=CONFIG['index_dir'],
    split='test'
)

print(f"   Train: {len(train_dataset):,}")
print(f"   Val:   {len(val_dataset):,}")
print(f"   Test:  {len(test_dataset):,}")

Loading datasets
Loading graph structure...
  Graph: 444,521 nodes, 734,274 edges
Loading labels from cache...
  Loaded 10466 suspicious / 434055 licit
  Split 'train': 311,164 samples
Loading graph structure...
  Graph: 444,521 nodes, 734,274 edges
Loading labels from cache...
  Loaded 10466 suspicious / 434055 licit
  Split 'val': 66,678 samples
Loading graph structure...
  Graph: 444,521 nodes, 734,274 edges
Loading labels from cache...
  Loaded 10466 suspicious / 434055 licit
  Split 'test': 66,679 samples
   Train: 311,164
   Val:   66,678
   Test:  66,679


## Create DataLoaders

In [4]:
from torch.utils.data import DataLoader

device = torch.device(CONFIG['device'])
print(f"Using device: {device}")

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=True if CONFIG['device'] == 'cuda' else False,
    collate_fn=FastEllipticDataset.collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True if CONFIG['device'] == 'cuda' else False,
    collate_fn=FastEllipticDataset.collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True if CONFIG['device'] == 'cuda' else False,
    collate_fn=FastEllipticDataset.collate_fn
)

edge_index = train_dataset.edge_index.to(device)

print(f" DataLoaders created:")
print(f"   Train batches: {len(train_loader):,}")
print(f"   Val batches:   {len(val_loader):,}")
print(f"   Test batches:  {len(test_loader):,}")

Using device: cpu
 DataLoaders created:
   Train batches: 1,216
   Val batches:   261
   Test batches:  261


## Create Model

In [5]:
# Create model
model_config = {
    **MODEL_CONFIG,
    'use_gnn': CONFIG['use_gnn'],
    'use_las': CONFIG['use_las'],
    'use_mamba': CONFIG['use_mamba'],
}

model = create_las_mamba_gnn(model_config).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f" Model created:")
print(f"   Parameters: {num_params:,}")
print(f"   Using GNN: {CONFIG['use_gnn']}")
print(f"   Using LAS: {CONFIG['use_las']}")
print(f"   Using Mamba: {CONFIG['use_mamba']}")

 Model created:
   Parameters: 364,996
   Using GNN: True
   Using LAS: True
   Using Mamba: True


## Create Optimizer and Loss

In [6]:
class_weights = torch.tensor(CONFIG['class_weights'], dtype=torch.float32, device=device)
criterion = get_loss_function(
    loss_type=CONFIG['loss_type'],
    class_weights=class_weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

print(f"Optimizer and Loss created:")
print(f"   Loss type: {CONFIG['loss_type']}")
print(f"   Class weights: {CONFIG['class_weights']}")
print(f"   Learning rate: {CONFIG['learning_rate']}")

Optimizer and Loss created:
   Loss type: focal
   Class weights: [1.0, 2000.0]
   Learning rate: 0.0001


## Training Functions

In [7]:
def train_epoch(model, loader, edge_index, criterion, optimizer, device, use_gnn=True, grad_clip_norm=0.5):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for batch_idx, (sequences, labels, batch_global_indices) in enumerate(loader):
        sequences = sequences.to(device)
        labels = labels.to(device)
        batch_global_indices = batch_global_indices.to(device)
        
        # Get local edge index for this batch
        if use_gnn and edge_index is not None and edge_index.shape[1] > 0:
            local_edge_index = get_local_edge_index(edge_index, batch_global_indices, device)
        else:
            local_edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
        
        node_features = sequences.mean(dim=(1, 2))
        
        optimizer.zero_grad()
        
        logits = model(node_features, sequences, local_edge_index)
        loss = criterion(logits, labels)
        
        if torch.isnan(loss):
            continue
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if batch_idx % 200 == 0:
            print(f"   Batch {batch_idx}/{len(loader)}, Loss: {loss.item():.4f}")
    
    return total_loss / num_batches if num_batches > 0 else 0.0


def evaluate(model, loader, edge_index, criterion, device, use_gnn=True, threshold=None):
    """Evaluate model"""
    if threshold is None:
        threshold = CONFIG['threshold']
    """Evaluate model"""
    model.eval()
    
    all_preds = []
    all_probs = []
    all_labels = []
    total_loss = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for sequences, labels, batch_global_indices in loader:
            sequences = sequences.to(device)
            labels = labels.to(device)
            batch_global_indices = batch_global_indices.to(device)
            
            if use_gnn and edge_index is not None and edge_index.shape[1] > 0:
                local_edge_index = get_local_edge_index(edge_index, batch_global_indices, device)
            else:
                local_edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
            
            node_features = sequences.mean(dim=(1, 2))
            
            logits = model(node_features, sequences, local_edge_index)
            
            loss_val = criterion(logits, labels)
            if not torch.isnan(loss_val):
                total_loss += loss_val.item()
                num_batches += 1
            
            all_preds.append(logits.argmax(dim=1).cpu())
            all_probs.append(torch.softmax(logits, dim=1)[:, 1].cpu())
            all_labels.append(labels.cpu())
    
    # Compute metrics
    metrics = compute_metrics(
        torch.cat(all_labels),
        torch.cat(all_preds),
        torch.cat(all_probs)
    )
    
    # Apply threshold tuning
    all_probs_tensor = torch.cat(all_probs)
    preds_with_threshold = (all_probs_tensor > threshold).long()
    metrics_threshold = compute_metrics(
        torch.cat(all_labels),
        preds_with_threshold,
        all_probs_tensor
    )
    
    if metrics_threshold['f1'] > metrics['f1']:
        print(f"   [DEBUG] Threshold={threshold} improved F1 from {metrics['f1']:.4f} to {metrics_threshold['f1']:.4f}")
        metrics['f1'] = metrics_threshold['f1']
        metrics['recall'] = metrics_threshold['recall']
        metrics['precision'] = metrics_threshold['precision']
    
    metrics['loss'] = total_loss / num_batches if num_batches > 0 else 0.0
    
    return metrics



## Training Loop

In [8]:
from pathlib import Path
import time

# Create checkpoint directory
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_f1': [],
    'val_auc_pr': [],
    'val_auc_roc': [],
    'val_recall': [],
    'val_precision': []
}

# Training loop
best_val_f1 = 0.0
patience_counter = 0


print("Starting training")


start_time = time.time()

for epoch in range(1, CONFIG['num_epochs'] + 1):
    epoch_start = time.time()
    
    # Train
    train_loss = train_epoch(
        model, train_loader, edge_index,
        criterion, optimizer, device,
        use_gnn=CONFIG['use_gnn']
    )
    
    # Validate
    if epoch % CONFIG['val_interval'] == 0:
        val_metrics = evaluate(
            model, val_loader, edge_index,
            criterion, device,
            use_gnn=CONFIG['use_gnn'],
            threshold=CONFIG['threshold']
        )
        
        # Update history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_f1'].append(val_metrics['f1'])
        history['val_auc_pr'].append(val_metrics.get('auc_pr', 0.0))
        history['val_auc_roc'].append(val_metrics.get('auc_roc', 0.0))
        history['val_recall'].append(val_metrics['recall'])
        history['val_precision'].append(val_metrics['precision'])
        
        # Print epoch results
        epoch_time = time.time() - epoch_start
        print(f"Epoch {epoch:03d} | " +
              f"train_loss={train_loss:.4f} | " +
              f"val_loss={val_metrics['loss']:.4f} | " +
              f"val_f1={val_metrics['f1']:.4f} | " +
              f"val_auc_pr={val_metrics.get('auc_pr', 0.0):.4f} | " +
              f"time={epoch_time:.1f}s")
        
        # Save best model
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            patience_counter = 0
            
            checkpoint_path = checkpoint_dir / 'best_model.pt'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_metrics': val_metrics,
                'config': CONFIG
            }, checkpoint_path)
            print(f"   -> Saved best model (val_f1={best_val_f1:.4f})")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= CONFIG['early_stopping_patience']:
            print(f"\nEarly stopping at epoch {epoch}")
            break
    else:
        print(f"Epoch {epoch:03d} | train_loss={train_loss:.4f}")

total_time = time.time() - start_time
print(f"Training completed in {total_time/60:.1f} minutes")
print(f"Best val_f1: {best_val_f1:.4f}")


Starting training


   Batch 0/1216, Loss: 20.1058
   Batch 200/1216, Loss: 1.6203
   Batch 400/1216, Loss: 1.3532
   Batch 600/1216, Loss: 0.9206
   Batch 800/1216, Loss: 1.2514
   Batch 1000/1216, Loss: 1.0149
   Batch 1200/1216, Loss: 0.9440
Epoch 001 | train_loss=1.7111 | val_loss=1.3801 | val_f1=0.0463 | val_auc_pr=0.1142 | time=1998.8s
   -> Saved best model (val_f1=0.0463)
   Batch 0/1216, Loss: 1.2155
   Batch 200/1216, Loss: 1.1759
   Batch 400/1216, Loss: 1.0993
   Batch 600/1216, Loss: 1.1389
   Batch 800/1216, Loss: 1.0286
   Batch 1000/1216, Loss: 1.0934
   Batch 1200/1216, Loss: 0.9667
Epoch 002 | train_loss=1.5687 | val_loss=1.5285 | val_f1=0.0537 | val_auc_pr=0.1058 | time=970.1s
   -> Saved best model (val_f1=0.0537)
   Batch 0/1216, Loss: 1.1343
   Batch 200/1216, Loss: 1.0044
   Batch 400/1216, Loss: 1.0181
   Batch 600/1216, Loss: 1.0312
   Batch 800/1216, Loss: 1.1679
   Batch 1000/1216, Loss: 1.0162
   Batch 1200/1216, Loss: 0.9089
Epoch 003 | train_loss=1.4870 | val_loss=1.3266 | va

KeyboardInterrupt: 

## Evaluate on Test Set

In [ ]:
# Load best model
checkpoint_path = checkpoint_dir / 'best_model.pt'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model from epoch {checkpoint['epoch']}")
    
    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_metrics = evaluate(
        model, test_loader, edge_index,
        criterion, device,
        use_gnn=CONFIG['use_gnn'],
        threshold=CONFIG['threshold']
    )
    
    print("TEST SET RESULTS")
    print(f"Loss:     {test_metrics['loss']:.4f}")
    print(f"Accuracy:{test_metrics['accuracy']:.4f}")
    print(f"Precision:{test_metrics['precision']:.4f}")
    print(f"Recall:   {test_metrics['recall']:.4f}")
    print(f"F1:       {test_metrics['f1']:.4f}")
    print(f"AUC-ROC:  {test_metrics.get('auc_roc', 0.0):.4f}")
    print(f"AUC-PR:   {test_metrics.get('auc_pr', 0.0):.4f}")
else:
    print("No checkpoint found!")

## Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

epochs_range = range(1, len(history['train_loss']) + 1)

# Plot Loss
axes[0, 0].plot(epochs_range, history['train_loss'], label='Train Loss')
if history['val_loss']:
    axes[0, 0].plot(epochs_range, history['val_loss'], label='Val Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Plot F1
if history['val_f1']:
    axes[0, 1].plot(epochs_range, history['val_f1'], label='Val F1', color='green')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('Validation F1 Score')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Plot AUC-PR
if history['val_auc_pr']:
    axes[1, 0].plot(epochs_range, history['val_auc_pr'], label='Val AUC-PR', color='orange')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('AUC-PR')
axes[1, 0].set_title('Validation AUC-PR')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Plot Recall and Precision
if history['val_recall']:
    axes[1, 1].plot(epochs_range, history['val_recall'], label='Recall', color='red')
if history['val_precision']:
    axes[1, 1].plot(epochs_range, history['val_precision'], label='Precision', color='blue')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_title('Validation Recall and Precision')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

print("Training history plot saved to 'training_history.png'")

## Summary

In [ ]:

print(f"Device: {device}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"Learning rate: {CONFIG['learning_rate']}")
print(f"Class weights: {CONFIG['class_weights']}")
print(f"Threshold: {CONFIG['threshold']}")
print(f"Epochs trained: {len(history['train_loss'])}")
print(f"Best F1: {best_val_f1:.4f}")
print(f"Total time: {total_time/60:.1f} minutes")
